# 🚀 2D → 3D Reconstruction Pipeline (NVIDIA 3D Master Plan Workflow)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dduy26/Img2d-to-3d/blob/P6-FullStack-Cloud/notebook/demo_colab.ipynb)

Hệ thống AI chuyển đổi ảnh 2D thành mô hình 3D hoàn chỉnh theo tiêu chuẩn NVIDIA:
- ⚡ **Chế độ 1: ĐƠN ẢNH (Single-View)** — Depth-Anything-V2-Small + Pinhole Grid Surface Mesh.
- 🌐 **Chế độ 2: ĐA ẢNH 360° (Multi-View)** — True Multi-View Silhouette Space Carving TSDF + Unified Canonical Frame + XAtlas PBR Texture Blending.
- 🏆 **Đầu Ra Duy Nhất:** 1 File 3D `.glb` kín nước 100% (Watertight Manifold, 0 cạnh biên hở, 1 khối liên thông), sẵn sàng in 3D (Cura / BambuStudio) và tương tác Web Three.js.

In [ ]:
# Cell 1: Clone Repository (nhánh P6-FullStack-Cloud) & Cài đặt môi trường
import os, sys, shutil

os.chdir('/content')
REPO = '/content/Img2d-to-3d'
BRANCH = 'P6-FullStack-Cloud'

if not os.path.isdir(REPO + '/.git'):
    shutil.rmtree(REPO, ignore_errors=True)
    !git clone -q --branch {BRANCH} https://github.com/dduy26/Img2d-to-3d.git {REPO}
else:
    !git -C {REPO} fetch -q origin
    !git -C {REPO} checkout -q {BRANCH}
    !git -C {REPO} reset --hard origin/{BRANCH}

os.chdir(REPO)
!git log --oneline -1

print('📦 Đang cài đặt các thư viện phụ thuộc...')
!pip install -q fastapi uvicorn python-multipart trimesh rembg onnxruntime networkx "scikit-image<0.26.0" opencv-python-headless transformers xatlas fast-simplification scipy

import trimesh
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
print(f'✅ Thư viện sẵn sàng: trimesh {trimesh.__version__} | Depth-Anything-V2-Small OK | cwd: {os.getcwd()}')


In [ ]:
# Cell 2: Chuẩn bị ảnh đầu vào (Chọn Chế độ: ĐƠN ẢNH hoặc ĐA ẢNH 360°)
# @title ⚙️ Lựa Chọn Chế Độ Dữ Liệu Đầu Vào { run: "auto" }
import os, glob, shutil
from PIL import Image
import matplotlib.pyplot as plt

INPUT_DIR = '/content/Img2d-to-3d/input'
ASSETS_DIR = '/content/Img2d-to-3d/notebook/assets/sample_objaverse_apple'
os.makedirs(INPUT_DIR, exist_ok=True)

CHE_DO_DAU_VAO = "Multi-view 360 (Mẫu Objaverse Apple 4 góc)" # @param ["Multi-view 360 (Mẫu Objaverse Apple 4 góc)", "Single-view (Mẫu Đơn Ảnh)", "Tự tải ảnh lên (Upload from Computer)", "Giữ nguyên ảnh hiện có trong input/"]

if CHE_DO_DAU_VAO == "Multi-view 360 (Mẫu Objaverse Apple 4 góc)":
    for f in glob.glob(f"{INPUT_DIR}/*"):
        if os.path.isfile(f): os.remove(f)
    for f in sorted(glob.glob(f"{ASSETS_DIR}/*.png")):
        shutil.copy(f, INPUT_DIR)
    print(f"✅ Đã nạp 4 ảnh mẫu chuẩn từ Objaverse (Front, Right, Back, Left).")

elif CHE_DO_DAU_VAO == "Single-view (Mẫu Đơn Ảnh)":
    for f in glob.glob(f"{INPUT_DIR}/*"):
        if os.path.isfile(f): os.remove(f)
    single_sample = os.path.join(ASSETS_DIR, "view_1_front.png")
    if os.path.exists(single_sample):
        shutil.copy(single_sample, os.path.join(INPUT_DIR, "sample_single.png"))
    print(f"✅ Đã nạp 1 ảnh mẫu đơn (Single-view mode).")

elif CHE_DO_DAU_VAO == "Tự tải ảnh lên (Upload from Computer)":
    from google.colab import files
    print("📤 Vui lòng chọn 1 ảnh (đơn ảnh) hoặc nhiều ảnh (đa góc) từ máy tính:")
    uploaded = files.upload()
    for fname in uploaded.keys():
        shutil.move(fname, os.path.join(INPUT_DIR, fname))
    print(f"✅ Đã tải lên {len(uploaded)} ảnh vào {INPUT_DIR}.")

# Tự động kiểm tra số lượng ảnh trong input/
existing_imgs = sorted([f for f in glob.glob(f"{INPUT_DIR}/*") if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])
num_imgs = len(existing_imgs)
assert num_imgs > 0, "Thư mục input/ đang trống! Vui lòng chọn chế độ nạp ảnh."

print("\n" + "=" * 60)
if num_imgs == 1:
    print(f"🎯 CHẾ ĐỘ PHÁT HIỆN: [SINGLE-VIEW (ĐƠN ẢNH)] (1 ảnh)")
else:
    print(f"🎯 CHẾ ĐỘ PHÁT HIỆN: [MULTI-VIEW 360° (ĐA ẢNH)] ({num_imgs} ảnh)")
print("=" * 60)

# Hiển thị Gallery ảnh xem trước
cols = min(6, num_imgs)
plt.figure(figsize=(3.5 * cols, 3.5))
for idx, img_path in enumerate(existing_imgs[:6]):
    img = Image.open(img_path)
    plt.subplot(1, cols, idx + 1)
    plt.imshow(img)
    plt.title(f"{os.path.basename(img_path)}\n{img.size} | {img.mode}", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()
print("Danh sách ảnh trong input/:", [os.path.basename(p) for p in existing_imgs])


In [ ]:
# Cell 3: Khởi chạy FastAPI Server & Cloudflare Tunnel (Tùy chọn truy cập Web UI)
import subprocess, time, os, re, urllib.request, sys

REPO = '/content/Img2d-to-3d'
BACKEND = os.path.join(REPO, 'notebook', 'backend')
os.chdir(REPO)

# Dọn dẹp tiến trình cũ
!pkill -f uvicorn || true
!pkill -f cloudflared || true
time.sleep(2)

# Tải cloudflared nếu chưa có
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

env = dict(os.environ)
env['PYTHONPATH'] = BACKEND + os.pathsep + env.get('PYTHONPATH', '')

print('🚀 Đang khởi động FastAPI Backend (app.py)...')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND, env=env, stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT, text=True,
)

# Health check chờ server nạp model
ready = False
for i in range(40):
    if server.poll() is not None:
        print('❌ Server thoát với mã lỗi:', server.returncode)
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2) as r:
            print(f'✅ Server READY ({i * 2}s):', r.read().decode())
            ready = True
            break
    except Exception:
        if i % 2 == 0 and i > 0:
            print(f'   ...đang nạp các mô hình ({i * 2}s)...')
        time.sleep(2)

# Bật Cloudflare Tunnel tạo link công khai
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT, text=True,
)

public_url = None
for _ in range(30):
    if os.path.exists('/content/tunnel.log'):
        log_text = open('/content/tunnel.log', encoding='utf-8', errors='replace').read()
        m = re.search(r'https://[\w.-]+\.trycloudflare\.com', log_text)
        if m:
            public_url = m.group(0)
            print('\n' + '=' * 60)
            print('🌐 LINK WEB UI (Xem 3D trực quan):', public_url)
            print('📘 LINK SWAGGER API DOCS        :', public_url + '/docs')
            print('=' * 60 + '\n')
            break
    time.sleep(1)


In [ ]:
# Cell 4: Thực thi Tái Tạo Mô Hình 3D Chuẩn NVIDIA (Single hoặc Multi-view)
import glob, os, sys, time, json
import trimesh

REPO = '/content/Img2d-to-3d'
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from notebook.backend.app import execute_3d_pipeline
from notebook.backend.engine_tsdf_mesh import mesh_health

imgs = sorted([f for f in glob.glob(f'{REPO}/input/*') if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])
assert len(imgs) > 0, 'Chưa có ảnh trong input/ -> vui lòng chạy lại Cell 2'

print(f'🚀 BẮT ĐẦU TÁI TẠO 3D VỚI {len(imgs)} ẢNH ĐẦU VÀO...')
for i, p in enumerate(imgs):
    print(f'  [{i+1}] {os.path.basename(p)}')

t0 = time.time()
res = execute_3d_pipeline(imgs)
total_time = time.time() - t0

print('\n' + '=' * 65)
print('📊 KẾT QUẢ TÁI TẠO 3D PIPELINE')
print('=' * 65)
print(f"Trạng thái (Status)   : {res.get('status')}")
print(f"Chế độ thực thi (Mode): {res.get('mode')}")
print(f"Pipeline triển khai   : {res.get('pipeline')}")
print(f"Tổng thời gian xử lý  : {total_time:.2f} giây")

out_file = res.get('output_file')
assert out_file and os.path.exists(out_file), f'Không tìm thấy file kết quả tại: {out_file}'
size_mb = os.path.getsize(out_file) / (1024 * 1024)
print(f'\n📦 FILE 3D .GLB DUY NHẤT ĐÃ XUẤT:')
print(f'   Đường dẫn: {out_file} ({size_mb:.2f} MB)')

# Đánh giá tiêu chuẩn hình học thực tế (Mesh Health)
print('\n' + '-' * 65)
print('🔬 KIỂM ĐỊNH HÌNH HỌC MESH HEALTH (TIÊU CHUẨN IN 3D & SLICER):')
print('-' * 65)
mesh_loaded = trimesh.load(out_file, force='mesh')
health = mesh_health(mesh_loaded)

print(f"  1. Kín nước (Watertight / Manifold) : {health['watertight']} (Bắt buộc True)")
print(f"  2. Cạnh biên hở (Boundary Edges)    : {health['boundary_edges']} (Bắt buộc 0)")
print(f"  3. Số khối liên thông (Components)  : {health['components']} (Bắt buộc 1)")
print(f"  4. Số mặt tam giác (Faces)          : {health['faces']}")
print(f"  5. Số đỉnh (Vertices)               : {health['vertices']}")
print(f"  6. Thể tích hình học (Volume)       : {health['volume']:.6f}")

if health['watertight'] and health['boundary_edges'] == 0 and health['components'] == 1:
    print('\n🎉 ĐẠT CHUẨN XUẤT SẮC 100%: Lưới 3D hoàn chỉnh, kín nước, không phần dư, sẵn sàng in 3D!')
else:
    print('\n⚠️ Cảnh báo: Lưới cần kiểm tra thêm.')


In [ ]:
# Cell 5: Trình xem 3D Interactive Three.js tích hợp trực tiếp & Nút Tải File .GLB
import glob, os, base64
from IPython.display import HTML, display

glbs = sorted(glob.glob('/content/Img2d-to-3d/output/*.glb'), key=os.path.getmtime, reverse=True)
if len(glbs) == 0:
    print('Chưa có file .glb nào trong output/. Vui lòng chạy Cell 4 trước.')
else:
    latest_glb = glbs[0]
    filename = os.path.basename(latest_glb)
    size_kb = os.path.getsize(latest_glb) / 1024
    
    with open(latest_glb, 'rb') as f:
        glb_b64 = base64.b64encode(f.read()).decode('utf-8')
    
    viewer_html = f'''
    <div style="width: 100%; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
      <div style="background: #0f172a; border-radius: 12px; padding: 16px; box-shadow: 0 10px 25px rgba(0,0,0,0.5);">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 12px;">
          <div>
            <span style="color: #38bdf8; font-weight: bold; font-size: 15px;">🎮 3D Model: {filename}</span>
            <span style="color: #94a3b8; font-size: 13px; margin-left: 8px;">({size_kb:.1f} KB)</span>
          </div>
          <div>
            <button id="wireframe-btn" style="background: #1e293b; color: #f8fafc; border: 1px solid #475569; padding: 6px 12px; border-radius: 6px; cursor: pointer; font-size: 12px; margin-right: 8px;">Bật Wireframe</button>
            <button id="download-btn" style="background: #0284c7; color: white; border: none; padding: 6px 14px; border-radius: 6px; cursor: pointer; font-size: 12px; font-weight: bold;">Tải File .GLB Về Máy</button>
          </div>
        </div>
        
        <div id="three-container" style="width: 100%; height: 500px; background: #020617; border-radius: 8px; position: relative; overflow: hidden;">
          <div style="position: absolute; bottom: 12px; left: 16px; color: #64748b; font-size: 12px; pointer-events: none;">
            🖱️ Chuột trái: Xoay 360° | Chuột phải: Di chuyển | Cuộn chuột: Phóng to/Thu nhỏ
          </div>
        </div>
      </div>
    </div>

    <script type="module">
      import * as THREE from 'https://unpkg.com/three@0.160.0/build/three.module.js';
      import {{ OrbitControls }} from 'https://unpkg.com/three@0.160.0/examples/jsm/controls/OrbitControls.js';
      import {{ GLTFLoader }} from 'https://unpkg.com/three@0.160.0/examples/jsm/loaders/GLTFLoader.js';
      
      const container = document.getElementById('three-container');
      const scene = new THREE.Scene();
      scene.background = new THREE.Color(0x0a0f1d);
      
      scene.add(new THREE.HemisphereLight(0xffffff, 0x334155, 2.0));
      const dirLight = new THREE.DirectionalLight(0xffffff, 1.5);
      dirLight.position.set(5, 10, 7);
      scene.add(dirLight);
      
      const dirLight2 = new THREE.DirectionalLight(0x38bdf8, 0.8);
      dirLight2.position.set(-5, -5, -5);
      scene.add(dirLight2);

      const gridHelper = new THREE.GridHelper(2, 20, 0x0284c7, 0x1e293b);
      gridHelper.position.y = 0;
      scene.add(gridHelper);
      
      const camera = new THREE.PerspectiveCamera(45, container.clientWidth / container.clientHeight, 0.01, 1000);
      const renderer = new THREE.WebGLRenderer({{ antialias: true, alpha: true }});
      renderer.setSize(container.clientWidth, container.clientHeight);
      renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));
      renderer.shadowMap.enabled = true;
      container.appendChild(renderer.domElement);
      
      const controls = new OrbitControls(camera, renderer.domElement);
      controls.enableDamping = true;
      controls.dampingFactor = 0.05;
      controls.autoRotate = true;
      controls.autoRotateSpeed = 2.0;

      let modelMesh = null;
      let isWireframe = false;
      
      const glbData = 'data:model/gltf-binary;base64,{glb_b64}';
      const loader = new GLTFLoader();
      loader.load(glbData, (gltf) => {{
        modelMesh = gltf.scene;
        modelMesh.traverse((o) => {{
          if (o.isMesh && o.material) {{
            if (o.geometry.attributes.color) o.material.vertexColors = true;
            o.material.side = THREE.DoubleSide;
            o.material.needsUpdate = true;
          }}
        }});
        
        const box = new THREE.Box3().setFromObject(modelMesh);
        const size = box.getSize(new THREE.Vector3()).length() || 1;
        const center = box.getCenter(new THREE.Vector3());
        
        modelMesh.position.x -= center.x;
        modelMesh.position.y -= box.min.y;
        modelMesh.position.z -= center.z;
        
        camera.position.set(0, size * 0.6, size * 1.5);
        controls.target.set(0, size * 0.3, 0);
        controls.update();
        scene.add(modelMesh);
      }});

      document.getElementById('wireframe-btn').addEventListener('click', () => {{
        isWireframe = !isWireframe;
        document.getElementById('wireframe-btn').textContent = isWireframe ? 'Tắt Wireframe' : 'Bật Wireframe';
        if (modelMesh) {{
          modelMesh.traverse((o) => {{
            if (o.isMesh && o.material) {{
              o.material.wireframe = isWireframe;
            }}
          }});
        }}
      }});

      document.getElementById('download-btn').addEventListener('click', () => {{
        const link = document.createElement('a');
        link.href = glbData;
        link.download = '{filename}';
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }});
      
      function animate() {{
        requestAnimationFrame(animate);
        controls.update();
        renderer.render(scene, camera);
      }}
      animate();

      window.addEventListener('resize', () => {{
        if (!container) return;
        camera.aspect = container.clientWidth / container.clientHeight;
        camera.updateProjectionMatrix();
        renderer.setSize(container.clientWidth, container.clientHeight);
      }});
    </script>
    '''
    display(HTML(viewer_html))
